In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install pymatgen matminer

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/883.4 kB ? eta -:--:--
   ---------------------------------------- 883.4/883.4 kB 18.5 MB/s  0:00:00
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ----------------------- ---------------- 3.1/5.3 MB 15.5 MB/s eta 0:00:01
   ------------------------------- -------- 4.2/5.3 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------- 5.3/5.3 MB 8.5 MB/s  0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   --------------- ------------------------ 4.2/11.1 MB 21.7 MB/s eta 0:00:01
   ---------------------- ----------------- 6.3/11.1 MB 16.3 MB/s eta 0:00:01

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To 

In [12]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

### Goal 1:
I want to see if given composition can predict classification

First I load the df
- dataset was obtained from the paper given (band-gap)

In [31]:
df = pd.read_excel('jz8b00124_si_003.xlsx')
print(df)

                    composition classification  regression Eg (eV)
0                Ag0.825Al0.175          metal                 NaN
1                  Ag0.82Al0.18          metal                 NaN
2                Ag0.818Al0.182          metal                 NaN
3                Ag0.803Al0.197          metal                 NaN
4                  Ag0.79Al0.21          metal                 NaN
...                         ...            ...                 ...
94090                  SbSeTeTl       nonmetal            0.250073
94091           Sb3.6SmSn0.4Ti3          metal                 NaN
94092             Se6Si2Ta15Te4          metal                 NaN
94093  Se0.05Sn0.95Te0.95Zn0.05       nonmetal            1.085840
94094         Sn0.53Te2TlZr0.47          metal                 NaN

[94095 rows x 3 columns]


**Note: when Eg = NaN, classification = metal**

### Materials Informatics

Now we want to convert our composition data into numerical features so that we can use them to train our ML model. 

In [32]:
df['composition'] = df['composition'].apply(Composition)

print(df)

            composition classification  regression Eg (eV)
0              (Ag, Al)          metal                 NaN
1              (Ag, Al)          metal                 NaN
2              (Ag, Al)          metal                 NaN
3              (Ag, Al)          metal                 NaN
4              (Ag, Al)          metal                 NaN
...                 ...            ...                 ...
94090  (Sb, Se, Te, Tl)       nonmetal            0.250073
94091  (Sb, Sm, Sn, Ti)          metal                 NaN
94092  (Se, Si, Ta, Te)          metal                 NaN
94093  (Se, Sn, Te, Zn)       nonmetal            1.085840
94094  (Sn, Te, Tl, Zr)          metal                 NaN

[94095 rows x 3 columns]


In [33]:
# A common preset featurizer
featurizer = ElementProperty.from_preset('magpie')

# Add numerical descriptor columns
df_feat = featurizer.featurize_dataframe(df, col_id="composition")

print(df_feat.head())
print(f"\nNumber of generated features: {len(featurizer.feature_labels())}")

ElementProperty: 100%|██████████| 94095/94095 [1:05:35<00:00, 23.91it/s]


  composition classification  regression Eg (eV)  MagpieData minimum Number  \
0    (Ag, Al)          metal                 NaN                       13.0   
1    (Ag, Al)          metal                 NaN                       13.0   
2    (Ag, Al)          metal                 NaN                       13.0   
3    (Ag, Al)          metal                 NaN                       13.0   
4    (Ag, Al)          metal                 NaN                       13.0   

   MagpieData maximum Number  MagpieData range Number  MagpieData mean Number  \
0                       47.0                     34.0                  41.050   
1                       47.0                     34.0                  40.880   
2                       47.0                     34.0                  40.812   
3                       47.0                     34.0                  40.302   
4                       47.0                     34.0                  39.860   

   MagpieData avg_dev Number  MagpieDa

In [34]:
df_feat.to_excel("features.xlsx", index=False)